[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/06-transformer-arch.ipynb)

# Transformer Architecture
**Module 7 — Lesson 6 | Estimated time: 40 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Describe the original "Attention is All You Need" architecture
- Implement sinusoidal positional encoding and visualise it
- Distinguish Pre-Norm from Post-Norm transformer blocks
- Build an encoder block with multi-head self-attention + feed-forward sublayer
- Build a decoder block adding cross-attention
- Assemble a full (small) Transformer encoder-decoder
- Train on a toy sequence-to-sequence task

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. The Transformer at a Glance

"Attention is All You Need" (Vaswani et al., 2017) replaced recurrent connections entirely with attention, enabling massive parallelism during training.

**Key components:**
- Token + Positional Embeddings
- Encoder: N × (Multi-Head Self-Attention → Feed-Forward)
- Decoder: N × (Masked Self-Attention → Cross-Attention → Feed-Forward)
- Final linear + softmax over vocabulary

All sublayers use residual connections and layer normalisation:
```
Post-Norm (original): x = LayerNorm(x + Sublayer(x))
Pre-Norm  (modern):   x = x + Sublayer(LayerNorm(x))
```

## 2. Positional Encoding

Since attention has no notion of order, we add a position-dependent signal to embeddings.

Sinusoidal PE (original paper):
$$PE_{(pos, 2i)}   = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

# Visualise
pe_layer = PositionalEncoding(d_model=64, max_len=50)
pe_vals = pe_layer.pe[0].numpy()  # (50, 64)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(pe_vals.T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.title('Positional Encoding Matrix')
plt.xlabel('Position'); plt.ylabel('Embedding dimension')

plt.subplot(1, 2, 2)
for dim in [0, 1, 4, 5, 16, 17]:
    plt.plot(pe_vals[:, dim], label=f'dim {dim}')
plt.title('PE values for selected dimensions')
plt.xlabel('Position'); plt.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 3. Feed-Forward Sublayer

Each encoder/decoder block contains a position-wise feed-forward network:
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

The inner dimension is typically 4× the model dimension (d_ff = 4 × d_model).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

print('FeedForward module created.')
ff = FeedForward(d_model=64, d_ff=256)
test_ff = ff(torch.randn(2, 10, 64))
print('Output shape:', test_ff.shape)

## 4. Encoder Block

In [ ]:
class EncoderBlock(nn.Module):
    """Pre-Norm Transformer encoder block."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff    = FeedForward(d_model, d_ff, dropout)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, src_key_padding_mask=None):
        # Self-attention sublayer
        x2 = self.norm1(x)
        attn_out, _ = self.attn(x2, x2, x2, key_padding_mask=src_key_padding_mask)
        x = x + self.drop(attn_out)
        # Feed-forward sublayer
        x = x + self.ff(self.norm2(x))
        return x

enc_block = EncoderBlock(d_model=64, num_heads=4, d_ff=256)
test_x = torch.randn(2, 12, 64)
print('Encoder block output:', enc_block(test_x).shape)

## 5. Decoder Block

In [ ]:
class DecoderBlock(nn.Module):
    """Pre-Norm Transformer decoder block with masked self-attn + cross-attn."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.self_attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        # Masked self-attention
        x2 = self.norm1(tgt)
        sa_out, _ = self.self_attn(x2, x2, x2, attn_mask=tgt_mask,
                                   key_padding_mask=tgt_key_padding_mask)
        tgt = tgt + self.drop(sa_out)
        # Cross-attention
        x2 = self.norm2(tgt)
        ca_out, _ = self.cross_attn(x2, memory, memory)
        tgt = tgt + self.drop(ca_out)
        # Feed-forward
        tgt = tgt + self.ff(self.norm3(tgt))
        return tgt

dec_block = DecoderBlock(d_model=64, num_heads=4, d_ff=256)
mem = torch.randn(2, 12, 64)
tgt = torch.randn(2, 8, 64)
causal = nn.Transformer.generate_square_subsequent_mask(8)
print('Decoder block output:', dec_block(tgt, mem, tgt_mask=causal).shape)

## 6. Full Transformer

In [ ]:
class SmallTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=64, num_heads=4,
                 num_layers=2, d_ff=256, max_len=64, dropout=0.1):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab, d_model, padding_idx=0)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)
        self.encoder   = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder   = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm_enc  = nn.LayerNorm(d_model)
        self.norm_dec  = nn.LayerNorm(d_model)
        self.fc_out    = nn.Linear(d_model, tgt_vocab)
        self.d_model   = d_model
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_padding_mask=None):
        x = self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model))
        for layer in self.encoder:
            x = layer(x, src_padding_mask)
        return self.norm_enc(x)

    def decode(self, tgt, memory, tgt_mask=None, tgt_padding_mask=None):
        x = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model))
        for layer in self.decoder:
            x = layer(x, memory, tgt_mask, tgt_padding_mask)
        return self.norm_dec(x)

    def forward(self, src, tgt):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        memory   = self.encode(src)
        dec_out  = self.decode(tgt, memory, tgt_mask)
        return self.fc_out(dec_out)

model = SmallTransformer(src_vocab=20, tgt_vocab=20, d_model=64, num_heads=4, num_layers=2)
src = torch.randint(1, 20, (4, 10))
tgt = torch.randint(1, 20, (4, 8))
out = model(src, tgt)
print('Transformer output:', out.shape)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 7. Train on Number Sorting (Toy Seq2Seq)

Task: given a sequence of integers (e.g. `[3, 1, 4, 1, 5]`), output the sorted sequence (`[1, 1, 3, 4, 5]`). This is a clean test of the transformer's sequence-to-sequence ability.

In [ ]:
# Tokens: 0=PAD, 1=SOS, 2=EOS, 3..12=numbers 0..9
PAD, SOS, EOS = 0, 1, 2
VOCAB = 13  # 0-2 special + 10 digit tokens

def make_sorting_batch(batch_size=64, seq_len=6):
    nums = np.random.randint(3, VOCAB, (batch_size, seq_len))
    src  = torch.tensor(nums, dtype=torch.long)
    sorted_nums = np.sort(nums, axis=1)
    sos_col = np.full((batch_size, 1), SOS)
    eos_col = np.full((batch_size, 1), EOS)
    tgt_in  = torch.tensor(np.hstack([sos_col, sorted_nums]), dtype=torch.long)  # teacher forcing input
    tgt_out = torch.tensor(np.hstack([sorted_nums, eos_col]), dtype=torch.long)  # expected output
    return src, tgt_in, tgt_out

# Verify shape
src, tgt_in, tgt_out = make_sorting_batch(4, 5)
print('src shape:    ', src.shape)
print('tgt_in shape: ', tgt_in.shape)
print('tgt_out shape:', tgt_out.shape)
print('Example src:   ', src[0].tolist())
print('Expected output:', tgt_out[0].tolist())

In [ ]:
model = SmallTransformer(src_vocab=VOCAB, tgt_vocab=VOCAB, d_model=64,
                         num_heads=4, num_layers=3, d_ff=256).to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=5e-4, betas=(0.9, 0.98))
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimiser, max_lr=5e-3, total_steps=600, pct_start=0.1)

losses = []
for step in range(600):
    model.train()
    src, tgt_in, tgt_out = make_sorting_batch(64, 6)
    src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
    optimiser.zero_grad()
    logits = model(src, tgt_in)       # (B, T, V)
    loss   = criterion(logits.reshape(-1, VOCAB), tgt_out.reshape(-1))
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimiser.step()
    scheduler.step()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f'Step {step:4d}: loss={loss.item():.3f}')

plt.plot(losses)
plt.title('Sorting Transformer Training Loss')
plt.xlabel('Step'); plt.ylabel('Cross-Entropy')
plt.tight_layout(); plt.show()

In [ ]:
# Greedy decoding inference
def greedy_decode(model, src, max_len=10):
    model.eval()
    src = src.unsqueeze(0).to(device)
    with torch.no_grad():
        memory = model.encode(src)
        dec_input = torch.tensor([[SOS]], device=device)
        result = []
        for _ in range(max_len):
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(dec_input.size(1)).to(device)
            out = model.decode(dec_input, memory, tgt_mask)
            logits  = model.fc_out(out[:, -1, :])
            next_id = logits.argmax(-1).item()
            if next_id == EOS: break
            result.append(next_id)
            dec_input = torch.cat([dec_input, torch.tensor([[next_id]], device=device)], dim=1)
    return result

# Test on 5 examples
print('Sorting test (token ids: 3..12 = digit 0..9):')
for _ in range(5):
    s = torch.randint(3, VOCAB, (6,))
    pred = greedy_decode(model, s)
    expected = sorted(s.tolist())
    mark = 'OK' if pred == expected else 'MISS'
    print(f'  Input: {s.tolist()}  Predicted: {pred}  Expected: {expected}  {mark}')

## Practice Exercises

**Exercise 1 — Pre-Norm vs Post-Norm**
Modify `EncoderBlock` to support Post-Norm (`x = LayerNorm(x + Sublayer(x))`) via an argument `pre_norm=True/False`. Train both variants on the sorting task for 600 steps and plot their loss curves side by side.

**Exercise 2 — Label Smoothing**
Replace `nn.CrossEntropyLoss()` with a label-smoothed variant using `nn.CrossEntropyLoss(label_smoothing=0.1)`. Observe whether the model's softmax outputs become less overconfident.

**Exercise 3 — Sinusoidal vs Learned PE**
Replace `PositionalEncoding` with a learned `nn.Embedding(max_len, d_model)`. Train both variants and compare sorting accuracy at 300, 600, and 1000 training steps. Which converges faster?